# Multi-Timeframe LSTM Training (IMPROVED - More Data)

## 🆕 Improvements Over v04:
- ✅ **Uses fillna() instead of dropna()** → Keeps ~2000 rows instead of ~250
- ✅ **Reduced SEQUENCE_LENGTH to 40** → More sequences
- ✅ **Removed SMA_50** → Less NaN values
- ✅ **Diagnostic output** at each step
- ✅ **Expected: 300-450 test samples** instead of 39

## Results from v04:
- Crash 500: 68.42% accuracy (but only 39 test samples ⚠️)
- **Goal**: Validate with 10x more test data

**Run all cells in order: Cell → Run All**

In [ ]:
# Cell 1: Imports and Configuration
import sys
import os
from pathlib import Path

# Change to project root directory
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
    os.chdir(project_root)
    print(f"Changed working directory to: {project_root}")
else:
    print(f"Current working directory: {Path.cwd()}")

sys.path.append(str(Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Configuration
SYMBOL = 'Crash 500 Index'
TIMEFRAMES = ['M15', 'H1', 'H4']
TARGET_TIMEFRAME = 'H4'

# IMPROVED: Reduced sequence length for more samples
SEQUENCE_LENGTH = 40  # Was 60
PREDICTION_HORIZON = 1
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.2
LEARNING_RATE = 0.001
BATCH_SIZE = 32
EPOCHS = 100
PATIENCE = 15

print("\n" + "="*60)
print("Multi-Timeframe LSTM (IMPROVED)")
print("="*60)
print(f"Symbol: {SYMBOL}")
print(f"Timeframes: {', '.join(TIMEFRAMES)}")
print(f"Sequence Length: {SEQUENCE_LENGTH} (reduced from 60)")
print(f"Goal: 300-450 test samples (vs 39 in v04)")
print("="*60)

In [ ]:
# Cell 2: Load Multi-Timeframe Data
from src.utils.helpers import load_data

def load_multi_timeframe_data(symbol, timeframes):
    data_dict = {}
    
    for tf in timeframes:
        df = load_data(symbol, tf)
        if df is None:
            print(f"❌ Failed to load {symbol} {tf}")
            return None
        data_dict[tf] = df
        print(f"✅ Loaded {symbol} {tf}: {len(df)} candles")
    
    return data_dict

print("Loading multi-timeframe data...\n")
data_dict = load_multi_timeframe_data(SYMBOL, TIMEFRAMES)

if data_dict:
    print("\n" + "="*60)
    print("Data Successfully Loaded")
    print("="*60)
    for tf, df in data_dict.items():
        print(f"{tf}: {len(df)} candles from {df.index[0]} to {df.index[-1]}")
    print("="*60)

In [ ]:
# Cell 3: Calculate Technical Indicators (IMPROVED - Less NaN)
def calculate_indicators(df, prefix=''):
    """Calculate technical indicators with reduced windows"""
    df = df.copy()
    initial_len = len(df)
    
    # Returns
    df[f'{prefix}Returns'] = df['Close'].pct_change()
    
    # IMPROVED: Reduced windows
    df[f'{prefix}SMA_10'] = df['Close'].rolling(window=10).mean()
    df[f'{prefix}SMA_20'] = df['Close'].rolling(window=20).mean()
    # Removed SMA_50 to reduce NaN
    
    df[f'{prefix}EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df[f'{prefix}EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df[f'{prefix}RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    exp1 = df['Close'].ewm(span=12, adjust=False).mean()
    exp2 = df['Close'].ewm(span=26, adjust=False).mean()
    df[f'{prefix}MACD'] = exp1 - exp2
    df[f'{prefix}MACD_signal'] = df[f'{prefix}MACD'].ewm(span=9, adjust=False).mean()
    
    # Bollinger Bands
    df[f'{prefix}BB_middle'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df[f'{prefix}BB_upper'] = df[f'{prefix}BB_middle'] + (bb_std * 2)
    df[f'{prefix}BB_lower'] = df[f'{prefix}BB_middle'] - (bb_std * 2)
    df[f'{prefix}BB_width'] = df[f'{prefix}BB_upper'] - df[f'{prefix}BB_lower']
    
    # ATR
    high_low = df['High'] - df['Low']
    high_close = np.abs(df['High'] - df['Close'].shift())
    low_close = np.abs(df['Low'] - df['Close'].shift())
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    true_range = np.max(ranges, axis=1)
    df[f'{prefix}ATR'] = true_range.rolling(14).mean()
    
    # Volatility
    df[f'{prefix}Volatility'] = df['Close'].rolling(window=20).std()
    
    nan_count = df.isna().sum().sum()
    print(f"  {prefix}: {initial_len} rows, {nan_count} NaN values")
    
    return df

print("Calculating technical indicators...\n")

for tf in TIMEFRAMES:
    prefix = f'{tf}_'
    data_dict[tf] = calculate_indicators(data_dict[tf], prefix=prefix)

print("\n✅ Technical Indicators Calculated")

In [ ]:
# Cell 4: Align Multi-Timeframe Data (IMPROVED - Uses fillna)
def align_multi_timeframe(data_dict, target_timeframe):
    """
    IMPROVED: Uses forward fill instead of dropna to keep more data
    """
    df_target = data_dict[target_timeframe].copy()
    print(f"\n📊 Starting alignment (target: {target_timeframe})")
    print(f"   Initial rows: {len(df_target)}")
    
    # Remove prefix from target
    target_prefix = f'{target_timeframe}_'
    df_target.columns = [col.replace(target_prefix, '') for col in df_target.columns]
    
    # Select features
    target_features = ['Close', 'Returns', 'SMA_10', 'SMA_20', 'EMA_10', 
                      'RSI', 'MACD', 'BB_middle', 'BB_width', 'ATR', 'Volatility']
    target_features = [f for f in target_features if f in df_target.columns]
    df_aligned = df_target[target_features].copy()
    
    # Merge other timeframes
    for tf in data_dict.keys():
        if tf == target_timeframe:
            continue
        
        df_tf = data_dict[tf].copy()
        tf_prefix = f'{tf}_'
        
        selected_features = [
            f'{tf_prefix}Close', f'{tf_prefix}Returns', f'{tf_prefix}SMA_10',
            f'{tf_prefix}RSI', f'{tf_prefix}MACD', f'{tf_prefix}Volatility'
        ]
        selected_features = [f for f in selected_features if f in df_tf.columns]
        df_tf_selected = df_tf[selected_features]
        
        df_aligned = pd.merge_asof(
            df_aligned.sort_index(),
            df_tf_selected.sort_index(),
            left_index=True,
            right_index=True,
            direction='backward'
        )
        print(f"   After merging {tf}: {len(df_aligned)} rows")
    
    # IMPROVED: Use fillna instead of dropna
    nan_before = df_aligned.isna().sum().sum()
    print(f"   NaN values before fill: {nan_before}")
    
    # Forward fill then backward fill
    df_aligned = df_aligned.fillna(method='ffill').fillna(method='bfill')
    
    nan_after = df_aligned.isna().sum().sum()
    print(f"   NaN values after fill: {nan_after}")
    
    # Only drop rows that still have NaN (should be very few)
    rows_before_drop = len(df_aligned)
    df_aligned = df_aligned.dropna()
    rows_dropped = rows_before_drop - len(df_aligned)
    
    print(f"\n✅ Alignment complete:")
    print(f"   Final rows: {len(df_aligned)}")
    print(f"   Rows dropped: {rows_dropped}")
    print(f"   Data retention: {len(df_aligned)/len(df_target)*100:.1f}%")
    
    return df_aligned

df_multi = align_multi_timeframe(data_dict, TARGET_TIMEFRAME)

print(f"\n📊 Multi-Timeframe Dataset: {len(df_multi)} rows × {len(df_multi.columns)} features")